# Week 4: Self-Organized Criticality — The BTW Sandpile

**Course:** From Conway to LangGraph — Agent Systems for Physicists  
**Topic:** Self-Organized Criticality and Power Laws  

**Lab Objectives:**
- Implement the Bak–Tang–Wiesenfeld sandpile model using vectorized NumPy
- Understand the transition from subcritical transient to critical steady state
- Measure avalanche size distributions and verify power-law behavior
- Learn proper power-law fitting (MLE, log-binning, KS test)
- Explore finite-size scaling and universality

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, Normalize
from matplotlib.animation import FuncAnimation
from scipy.stats import linregress
from scipy.ndimage import label
from IPython.display import HTML
import warnings
warnings.filterwarnings('ignore')

# Course color palette
COLOR_BG = "#1A1A1A"
COLOR_A = "#4A9ECC"
COLOR_B = "#E8B931"
COLOR_GREEN = "#7ECE8B"
COLOR_ORANGE = "#D19A66"
COLOR_PURPLE = "#C678DD"

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': COLOR_BG,
    'axes.facecolor': '#2D2D2D',
    'axes.edgecolor': '#444444',
    'text.color': '#FFFFFF',
    'axes.labelcolor': '#CCCCCC',
    'xtick.color': '#999999',
    'ytick.color': '#999999',
    'grid.color': '#444444',
    'font.size': 12,
})

print("Setup complete ✓")

---
## Part 1: From Tuned to Self-Organized Criticality

Last week we saw that the Schelling model has a **phase transition** at a critical tolerance $T_c \approx 0.30$. To observe criticality (power-law correlations, divergent fluctuations), we had to **tune** $T$ to this special value.

But in nature, power laws appear everywhere — earthquakes, forest fires, neural avalanches — without anyone tuning a parameter. **Self-Organized Criticality (SOC)** explains how:

> Some driven dissipative systems naturally evolve toward a critical state — the critical point is an **attractor** of the dynamics, not a special parameter value.

**Three ingredients of SOC:**
1. **Slow driving** — energy/mass added gradually
2. **Threshold dynamics** — local instabilities trigger cascading events
3. **Dissipation at boundaries** — energy leaves the system (open boundaries)

---
## Part 2: The BTW Sandpile Model

### 2.1 Model Definition (Bak, Tang & Wiesenfeld, 1987)

**Grid:** Square lattice $N \times N$ with **open boundaries** (grains fall off the edge).

**State:** Height $z(x,y) \in \{0, 1, 2, 3, \ldots\}$ — number of sand grains at each site.

**Critical height:** $z_c = 4$ (= number of von Neumann neighbors in 2D).

**Dynamics:**
1. **Drive:** Add one grain at a random site: $z(x,y) \to z(x,y) + 1$
2. **Relax:** While any $z(x,y) \geq z_c$:
   - $z(x,y) \to z(x,y) - 4$
   - $z(\text{neighbors}) \to z(\text{neighbors}) + 1$ (grains at boundary are lost)
3. **Measure:** Record avalanche size $s$ = total number of topplings

### 2.2 Key Properties

- **Abelian property** (Dhar, 1990): The final state after an avalanche is independent of the toppling order.
- **The sandpile is a threshold CA**, not an ABM — that's why we use NumPy (like Weeks 1–2), not Mesa.
- **Open boundaries** provide dissipation: grains that "topple off" the edge are lost forever.

### 2.3 NumPy Implementation

The toppling rule is essentially the **discrete Laplacian** applied to unstable sites — the same array-slicing trick we used for Conway's Game of Life.

In [ ]:
class Sandpile:
    """BTW Abelian sandpile on N×N grid with open boundaries."""

    def __init__(self, N, z_crit=4):
        self.N = N
        self.z_crit = z_crit
        self.grid = np.zeros((N, N), dtype=int)

    def add_grain(self, x=None, y=None):
        """Add one grain at (x, y). Random site if not specified."""
        if x is None:
            x, y = np.random.randint(self.N, size=2)
        self.grid[x, y] += 1

    def relax(self):
        """Topple all unstable sites until stable. Return avalanche size."""
        total_topplings = 0
        while True:
            unstable = self.grid >= self.z_crit
            if not unstable.any():
                break
            n_topple = unstable.sum()
            total_topplings += n_topple

            # Remove 4 grains from each unstable site
            self.grid[unstable] -= self.z_crit

            # Distribute 1 grain to each of 4 neighbors
            # Open boundaries: grains at edges are simply lost
            self.grid[1:, :]  += unstable[:-1, :]  # from above
            self.grid[:-1, :] += unstable[1:, :]   # from below
            self.grid[:, 1:]  += unstable[:, :-1]   # from left
            self.grid[:, :-1] += unstable[:, 1:]    # from right

        return total_topplings

    def step(self):
        """One full step: add grain + relax. Return avalanche size."""
        self.add_grain()
        return self.relax()

    def total_grains(self):
        return self.grid.sum()

    def mean_height(self):
        return self.grid.mean()


# Quick test
sp = Sandpile(16)
sizes = [sp.step() for _ in range(5000)]
print(f"Grid size: {sp.N}×{sp.N}")
print(f"Mean height: {sp.mean_height():.2f}")
print(f"Max height: {sp.grid.max()}")
print(f"Max avalanche: {max(sizes)}")
print(f"Fraction with size 0: {sizes.count(0)/len(sizes):.1%}")
print("Sandpile class ready ✓")

---
## Part 3: Visualization

### 3.1 The Height Field

In the critical steady state, sites have heights $z \in \{0, 1, 2, 3\}$ (since $z \geq 4$ topples immediately during relaxation).

In [ ]:
# Build a sandpile to steady state
sp = Sandpile(64)
# Warmup: drive to critical state
for _ in range(50000):
    sp.step()

# Custom colormap: dark to warm (sand-like)
sand_cmap = ListedColormap(['#1A1A1A', '#3A3520', '#7A6830', '#D4A840'])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Height field
im = axes[0].imshow(sp.grid, cmap=sand_cmap, interpolation='nearest',
                     vmin=0, vmax=3)
axes[0].set_title('Height field z(x,y)', fontweight='bold')
axes[0].set_xticks([])
axes[0].set_yticks([])
plt.colorbar(im, ax=axes[0], ticks=[0, 1, 2, 3], label='Height')

# Height histogram
counts = [np.sum(sp.grid == h) for h in range(4)]
colors = ['#1A1A1A', '#3A3520', '#7A6830', '#D4A840']
axes[1].bar([0, 1, 2, 3], counts, color=colors, edgecolor='#666666')
axes[1].set_xlabel('Height z')
axes[1].set_ylabel('Number of sites')
axes[1].set_title('Height distribution', fontweight='bold')
axes[1].set_xticks([0, 1, 2, 3])

# Mean height over time
sp2 = Sandpile(64)
heights_over_time = []
for i in range(60000):
    sp2.step()
    if i % 100 == 0:
        heights_over_time.append(sp2.mean_height())

axes[2].plot(np.arange(len(heights_over_time)) * 100, heights_over_time,
             color=COLOR_B, linewidth=0.8)
axes[2].set_xlabel('Grains added')
axes[2].set_ylabel('Mean height ⟨z⟩')
axes[2].set_title('Approach to critical state', fontweight='bold')
axes[2].axhline(y=heights_over_time[-1], color=COLOR_A, linestyle='--',
                alpha=0.5, label=f'Steady state ≈ {heights_over_time[-1]:.2f}')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nSteady-state mean height: {sp.mean_height():.3f}")
print(f"Height distribution: {dict(zip(range(4), counts))}")

### 3.2 Visualizing a Single Avalanche

Let's watch one avalanche unfold step by step.

In [ ]:
# ── Cell A: distribution of discarded avalanche sizes ─────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Re-run the search loop but record every attempt (not just the winner)
sp_diag = SandpileWithHistory(32)
for _ in range(20_000):          # bring to steady state
    sp_diag.step()

np.random.seed(42)
N_ATTEMPTS = 2000
sizes = []

for _ in range(N_ATTEMPTS):
    grid_backup = sp_diag.grid.copy()
    sp_diag.add_grain()
    _, s = sp_diag.relax_with_history()
    sizes.append(s)
    if s < 50:                   # restore if not interesting
        sp_diag.grid = grid_backup
        sp_diag.add_grain()
        sp_diag.relax()

sizes = np.array(sizes)
n_trivial  = (sizes == 0).sum()   # grain lands on stable site, no cascade
n_small    = ((sizes > 0) & (sizes < 50)).sum()
n_large    = (sizes >= 50).sum()

print(f"Attempts           : {N_ATTEMPTS}")
print(f"Trivial (size=0)   : {n_trivial}  ({100*n_trivial/N_ATTEMPTS:.1f}%)")
print(f"Small   (0<s<50)   : {n_small}    ({100*n_small/N_ATTEMPTS:.1f}%)")
print(f"Large   (s≥50)     : {n_large}    ({100*n_large/N_ATTEMPTS:.1f}%)")
print(f"Mean size          : {sizes.mean():.2f}  |  Median: {np.median(sizes):.0f}")

# ── Figure ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: linear histogram
ax = axes[0]
bins_lin = np.linspace(0, sizes.max(), 60)
ax.hist(sizes[sizes > 0], bins=bins_lin, color='#4A9ECC', edgecolor='white',
        linewidth=0.4, label='size > 0')
ax.axvline(50, color='#E8B931', linewidth=1.5, linestyle='--', label='threshold = 50')
ax.set_xlabel('Avalanche size  (topplings)')
ax.set_ylabel('Count')
ax.set_title('Linear scale')
ax.legend(fontsize=9)
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# Right: log-log histogram → power-law signature
ax = axes[1]
pos = sizes[sizes > 0]
log_bins = np.logspace(np.log10(pos.min()), np.log10(pos.max()), 40)
counts, edges = np.histogram(pos, bins=log_bins)
centers = np.sqrt(edges[:-1] * edges[1:])        # geometric midpoint
mask = counts > 0
ax.scatter(centers[mask], counts[mask], s=18, color='#4A9ECC', zorder=3)

# Fit a power law P(s) ∝ s^{-τ} on the tail (s ≥ 5)
tail = pos[pos >= 5]
if len(tail) > 10:
    tau, _ = np.polyfit(np.log(tail), np.zeros(len(tail)), 0), None
    # Use MLE for the exponent (more robust than log-log OLS)
    tau_mle = 1 + len(tail) / np.log(tail / tail.min()).sum()
    xs = np.logspace(np.log10(5), np.log10(pos.max()), 100)
    # Normalise to data
    c = counts[mask].max() * (centers[mask].max() ** tau_mle)
    ax.plot(xs, c * xs**(-tau_mle), color='#E8B931', linewidth=1.5,
            label=f'MLE slope  τ ≈ {tau_mle:.2f}')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Avalanche size  s')
ax.set_ylabel('Count')
ax.set_title('Log–log  (power-law check)')
ax.legend(fontsize=9)
ax.grid(True, which='both', alpha=0.2)

fig.suptitle(f'Avalanche-size distribution  —  {N_ATTEMPTS} attempts on 32×32 grid',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell B: improved avalanche visualisation for Colab ────────────────────
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.cm import ScalarMappable
import matplotlib.gridspec as gridspec

# Re-use history and size from the search above (or re-run the search cell)
# history, size = ...  (assumed already in scope)

n_show  = min(6, len(history) - 1)        # exclude the final stable frame
indices = np.linspace(0, len(history) - 2, n_show, dtype=int)

# ── Layout: n_show panels + 1 narrow colorbar column ───────────────────────
fig = plt.figure(figsize=(3.0 * n_show + 0.6, 4.2))
gs  = gridspec.GridSpec(
    1, n_show + 1,
    width_ratios=[1] * n_show + [0.07],
    wspace=0.06,
    left=0.04, right=0.96, top=0.88, bottom=0.06,
)

# Height colormap: 0–3 are stable levels, 4+ are toppling artefacts
height_cmap = plt.cm.YlOrRd
vmin_h, vmax_h = 0, 5

# Temporal colourmap for the colourbar (round index → colour)
n_rounds = len(history) - 1
t_cmap   = plt.cm.get_cmap('plasma', n_show)

for col, idx in enumerate(indices):
    ax = fig.add_subplot(gs[0, col])
    grid_snap, unstable = history[idx]

    # Layer 1 — height map
    ax.imshow(grid_snap, cmap=height_cmap, interpolation='nearest',
              vmin=vmin_h, vmax=vmax_h, aspect='equal')

    # Layer 2 — toppling sites in red with temporal tint
    if unstable.any():
        t_color = t_cmap(col / max(n_show - 1, 1))
        overlay = np.ma.masked_where(~unstable, np.ones_like(grid_snap, float))
        ax.imshow(overlay,
                  cmap=ListedColormap([t_color]),
                  interpolation='nearest',
                  alpha=0.72,
                  aspect='equal')

    # Round label + toppling count for this round
    n_toppling = unstable.sum() if idx < len(history) - 1 else 0
    round_label = f'Round {idx}\n({n_toppling} sites)'
    ax.set_title(round_label, fontsize=9, pad=3)
    ax.set_xticks([])
    ax.set_yticks([])

    # Thin coloured border matching the temporal colour
    for spine in ax.spines.values():
        spine.set_edgecolor(t_cmap(col / max(n_show - 1, 1)))
        spine.set_linewidth(2.0)

# ── Temporal colourbar (right column) ──────────────────────────────────────
ax_cb = fig.add_subplot(gs[0, -1])
sm    = ScalarMappable(
    cmap=t_cmap,
    norm=BoundaryNorm(np.arange(n_show + 1), n_show),
)
sm.set_array([])
cb = fig.colorbar(sm, cax=ax_cb)
cb.set_label('Frame (round index)', fontsize=8, labelpad=4)
tick_positions = np.arange(n_show) + 0.5
cb.set_ticks(tick_positions)
cb.set_ticklabels([f'r{i}' for i in indices], fontsize=7)

# ── Height colorbar below the panels ───────────────────────────────────────
sm_h = ScalarMappable(cmap=height_cmap,
                      norm=plt.Normalize(vmin=vmin_h, vmax=vmax_h))
sm_h.set_array([])
cax_h = fig.add_axes([0.04, 0.00, 0.86, 0.03])   # [left, bottom, width, height]
cb_h  = fig.colorbar(sm_h, cax=cax_h, orientation='horizontal')
cb_h.set_label('Height z', fontsize=8)
cb_h.set_ticks([0, 1, 2, 3, 4, 5])

# ── Main title ──────────────────────────────────────────────────────────────
fig.suptitle(
    f'Avalanche  —  {size} topplings over {len(history)-1} rounds\n'
    f'Grid 32×32  ·  coloured border = temporal order  ·  red overlay = active sites',
    fontsize=11, fontweight='bold', y=0.99,
)

plt.savefig('avalanche_vis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → avalanche_vis.png")

---
## Part 4: Avalanche Statistics

Now let's collect a large sample of avalanches and study their distribution.

### 4.1 Collecting Data

In [ ]:
def run_sandpile(N=64, n_grains=100000, warmup=10000, seed=None):
    """Run sandpile, return avalanche sizes after warmup."""
    if seed is not None:
        np.random.seed(seed)
    sp = Sandpile(N)
    avalanches = []

    for i in range(n_grains):
        size = sp.step()
        if i >= warmup:
            avalanches.append(size)

    return sp, np.array(avalanches)


print("Running sandpile (N=64, 100k grains)...")
sp, avalanches = run_sandpile(N=64, n_grains=100000, warmup=10000, seed=42)
print(f"Done ✓")
print(f"Avalanches collected: {len(avalanches)}")
print(f"Non-zero avalanches: {np.sum(avalanches > 0)} ({np.mean(avalanches > 0):.1%})")
print(f"Max size: {avalanches.max()}")
print(f"Mean size: {avalanches.mean():.2f}")
print(f"Mean height: {sp.mean_height():.3f}")

### 4.2 Log-Binned Histogram

Power-law distributions span many orders of magnitude. Linear bins waste resolution at large $s$ (few events, wide bins) and over-resolve at small $s$. **Logarithmic binning** gives equal weight to each decade.

In [ ]:
def log_binned_histogram(data, n_bins=30):
    """Create logarithmically-spaced histogram.

    Returns bin centers (geometric mean) and probability density.
    """
    data_pos = data[data > 0]
    if len(data_pos) == 0:
        return np.array([]), np.array([])

    bins = np.logspace(
        np.log10(data_pos.min()),
        np.log10(data_pos.max()),
        n_bins + 1
    )
    counts, edges = np.histogram(data_pos, bins=bins)
    # Geometric mean of bin edges
    centers = np.sqrt(edges[:-1] * edges[1:])
    # Normalize: probability density
    widths = edges[1:] - edges[:-1]
    density = counts / (widths * len(data_pos))

    mask = counts > 0
    return centers[mask], density[mask]


# Plot
centers, density = log_binned_histogram(avalanches, n_bins=35)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Linear axes
ax1.hist(avalanches[avalanches > 0], bins=100, color=COLOR_A, alpha=0.7,
         edgecolor='none')
ax1.set_xlabel('Avalanche size s')
ax1.set_ylabel('Count')
ax1.set_title('Linear histogram (misleading!)', fontweight='bold')
ax1.set_xlim(0, 500)

# Log-log
ax2.loglog(centers, density, 'o', color=COLOR_A, markersize=4, alpha=0.8)
ax2.set_xlabel('Avalanche size s')
ax2.set_ylabel('P(s)')
ax2.set_title('Log-binned histogram (correct)', fontweight='bold')
ax2.grid(True, alpha=0.3, which='both')

# Guide line
s_ref = np.logspace(0.5, 2.5, 50)
ax2.plot(s_ref, 0.5 * s_ref**(-1.27), '--', color=COLOR_B, linewidth=2,
         label=r'$s^{-1.27}$ (reference)')
ax2.legend(fontsize=12)

plt.tight_layout()
plt.show()

### 4.3 Maximum Likelihood Estimation (MLE)

**Never** fit a power law by doing linear regression on a log-log histogram. The correct approach (Clauset, Shalizi & Newman, 2009) is **Maximum Likelihood Estimation**:

For a discrete power law $P(s) \propto s^{-\tau}$ with $s \geq s_{\min}$:

$$\hat{\tau} = 1 + n \left[ \sum_{i=1}^{n} \ln \frac{s_i}{s_{\min} - 0.5} \right]^{-1}$$

where $n$ is the number of observations with $s \geq s_{\min}$.

In [ ]:
def mle_power_law(data, s_min):
    """MLE estimate of power-law exponent tau for discrete data.

    Uses the formula from Clauset, Shalizi & Newman (2009).
    """
    data_above = data[data >= s_min]
    n = len(data_above)
    if n == 0:
        return np.nan, 0
    tau = 1 + n / np.sum(np.log(data_above / (s_min - 0.5)))
    # Standard error
    sigma = (tau - 1) / np.sqrt(n)
    return tau, sigma


def find_s_min_ks(data, s_min_range=None):
    """Find optimal s_min by minimizing KS distance."""
    if s_min_range is None:
        unique_vals = np.unique(data[data > 0])
        s_min_range = unique_vals[:min(50, len(unique_vals))]

    best_ks = np.inf
    best_smin = s_min_range[0]
    best_tau = 2.0

    for s_min in s_min_range:
        tail = data[data >= s_min]
        if len(tail) < 30:  # need enough data
            continue
        tau, _ = mle_power_law(data, s_min)
        if np.isnan(tau) or tau <= 1:
            continue

        # Compute KS statistic
        sorted_tail = np.sort(tail)
        n = len(sorted_tail)
        empirical_cdf = np.arange(1, n + 1) / n
        # Theoretical CDF for discrete power law (approximation)
        theoretical_cdf = 1 - (sorted_tail / s_min) ** (1 - tau)
        theoretical_cdf = np.clip(theoretical_cdf, 0, 1)
        ks = np.max(np.abs(empirical_cdf - theoretical_cdf))

        if ks < best_ks:
            best_ks = ks
            best_smin = s_min
            best_tau = tau

    return best_smin, best_tau, best_ks


# Fit
aval_pos = avalanches[avalanches > 0]
s_min_opt, tau_opt, ks_opt = find_s_min_ks(aval_pos)
tau_fixed, sigma_fixed = mle_power_law(aval_pos, s_min=1)

print(f"MLE fit (s_min = 1):  τ = {tau_fixed:.3f} ± {sigma_fixed:.3f}")
print(f"Optimal s_min = {s_min_opt}:  τ = {tau_opt:.3f}  (KS = {ks_opt:.4f})")
print(f"Literature value: τ ≈ 1.27 (2D BTW sandpile)")

In [ ]:
# Plot with MLE fit
fig, ax = plt.subplots(figsize=(9, 6))

centers, density = log_binned_histogram(aval_pos, n_bins=35)
ax.loglog(centers, density, 'o', color=COLOR_A, markersize=5, alpha=0.7,
          label='Data (log-binned)')

# MLE fit line
s_fit = np.logspace(np.log10(s_min_opt), np.log10(aval_pos.max()), 50)
# Normalize: P(s) = C * s^(-tau) where C is chosen to match at s_min
idx_near = np.argmin(np.abs(centers - s_min_opt))
C = density[idx_near] * centers[idx_near]**tau_opt
ax.plot(s_fit, C * s_fit**(-tau_opt), '--', color=COLOR_B, linewidth=2,
        label=f'MLE: $\\tau$ = {tau_opt:.2f} ($s_{{min}}$ = {s_min_opt})')

ax.axvline(x=s_min_opt, color=COLOR_GREEN, linestyle=':', alpha=0.5,
           label=f'$s_{{min}}$ = {s_min_opt}')

ax.set_xlabel('Avalanche size s', fontsize=13)
ax.set_ylabel('P(s)', fontsize=13)
ax.set_title('Avalanche Size Distribution with MLE Fit', fontweight='bold',
             fontsize=15)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

---
## Part 5: Transient to Criticality

Starting from an empty grid, how long does it take to reach the critical state?

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for N, color in zip([16, 32, 64], [COLOR_A, COLOR_B, COLOR_GREEN]):
    sp = Sandpile(N)
    heights = []
    running_mean_size = []
    window = []

    n_steps = N * N * 20  # Scale with system size
    sample_every = max(1, n_steps // 500)

    for i in range(n_steps):
        size = sp.step()
        window.append(size)
        if len(window) > 500:
            window.pop(0)

        if i % sample_every == 0:
            heights.append(sp.mean_height())
            running_mean_size.append(np.mean(window))

    x = np.arange(len(heights)) * sample_every
    ax1.plot(x, heights, color=color, linewidth=1, label=f'N = {N}')
    ax2.plot(x, running_mean_size, color=color, linewidth=1, label=f'N = {N}')

ax1.set_xlabel('Grains added')
ax1.set_ylabel('Mean height ⟨z⟩')
ax1.set_title('Mean height: transient → steady state', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Grains added')
ax2.set_ylabel('Running mean avalanche size')
ax2.set_title('Mean avalanche size', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("The transient length scales as ~ N² (filling the grid).")
print("After the transient, both quantities fluctuate around a constant.")

---
## Part 6: Animation

Watch the sandpile grow from empty to critical.

In [ ]:
def animate_sandpile(N=32, n_frames=200, grains_per_frame=50,
                     interval=80):
    sp = Sandpile(N)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5),
                                    gridspec_kw={'width_ratios': [1.2, 1]})

    im = ax1.imshow(sp.grid, cmap='YlOrRd', interpolation='nearest',
                    vmin=0, vmax=5)
    ax1.set_xticks([])
    ax1.set_yticks([])
    title = ax1.set_title('Grains: 0', fontweight='bold')
    plt.colorbar(im, ax=ax1, label='Height')

    height_history = []
    line, = ax2.plot([], [], color=COLOR_B, linewidth=1.5)
    ax2.set_xlim(0, n_frames * grains_per_frame)
    ax2.set_ylim(0, 3)
    ax2.set_xlabel('Grains added')
    ax2.set_ylabel('Mean height')
    ax2.set_title('Approach to criticality', fontweight='bold')
    ax2.grid(True, alpha=0.3)

    def update(frame):
        for _ in range(grains_per_frame):
            sp.step()
        im.set_data(sp.grid)
        total = (frame + 1) * grains_per_frame
        title.set_text(f'Grains: {total}')
        height_history.append(sp.mean_height())
        x_data = np.arange(len(height_history)) * grains_per_frame
        line.set_data(x_data, height_history)
        return im, title, line

    anim = FuncAnimation(fig, update, frames=n_frames,
                         interval=interval, blit=False)
    plt.tight_layout()
    return anim




anim = animate_sandpile(N=32, n_frames=150, grains_per_frame=80,
                        interval=80)
HTML(anim.to_jshtml())

In [ ]:
anim = animate_sandpile(N=256, n_frames=150, grains_per_frame=1500,
                        interval=80)
HTML(anim.to_jshtml())

---
## Part 7: Comparison of the Three Paradigms

Let's step back and see where the sandpile fits in our course journey.

| | CA (Weeks 1–2) | ABM (Week 3) | SOC (Week 4) |
|---|---|---|---|
| **Entities** | Cells | Agents | Cells (threshold CA) |
| **Update** | Deterministic rule | Utility-based | Threshold-driven |
| **Boundaries** | Periodic | Periodic (torus) | Open (dissipative) |
| **Criticality** | Rule-dependent | Tuned ($T = T_c$) | **Self-organized** |
| **Key measure** | Entropy | Segregation $S$ | Avalanche size $P(s)$ |
| **Signature** | Complexity classes | Phase transition | Power laws |
| **Tool** | NumPy | Mesa | NumPy |

The sandpile is a CA with a twist: open boundaries + slow driving create a fundamentally new phenomenon (SOC) that isn't possible in standard CA with periodic boundaries.

---
## Exercises

### Exercise 1: Avalanche Size Distribution (Core, ~45 min)

Run the sandpile for $N = 64$, 100,000 grains (10,000 warmup). Plot the avalanche size distribution with log-binning. Fit the exponent $\tau$ using MLE. Compare with the accepted value $\tau \approx 1.27$.

*Bonus: Try also a naive least-squares fit on the log-log histogram. Compare the two estimates.*

In [ ]:
# YOUR CODE HERE
# ...


### Exercise 2: Finite-Size Scaling (Core, ~60 min)

Run for $N = 16, 32, 64, 128$. Plot $P(s)$ for all sizes on the same log-log axes. Show that the cutoff grows with $N$.

Try to collapse the curves using the scaling ansatz $P(s) = s^{-\tau} f(s/N^D)$ where $D$ is a fractal dimension. You can estimate $D$ by plotting $s_{\max}$ vs $N$ on a log-log plot.

*Hint: $s_{\max} \propto N^D$ with $D \approx 2.7$ for the 2D BTW model.*

In [ ]:
# YOUR CODE HERE
# ...


### Exercise 3: Transient to Criticality (Core, ~45 min)

Starting from an empty grid, record avalanche sizes over time. Measure:
1. The warmup time $t_w$ (when mean height stabilizes) as a function of $N$.
2. How $t_w$ scales with $N$ (plot on log-log, fit slope).
3. Compare the avalanche size distribution during transient vs steady state.

In [ ]:
# YOUR CODE HERE
# ...


### Exercise 4: Avalanche Duration and Scaling Relations (Advanced, ~60 min)

Modify the `relax` method to also return:
- **Duration** $t$ = number of toppling rounds
- **Area** $a$ = number of distinct sites that toppled at least once

Then:
1. Plot $P(t)$ and fit the exponent $\tau_t$.
2. Plot $\langle s \rangle$ vs $t$ and measure the exponent $\gamma$ in $s \propto t^\gamma$.
3. Check the scaling relation: $\gamma = (\tau_t - 1)/(\tau_s - 1)$.

In [ ]:
# YOUR CODE HERE
# ...


---
## Solutions (mirko: by Claude NOT cecked by me !!)

### Solution 1: Avalanche Size Distribution

In [ ]:
# Collect data
print("Running sandpile N=64...")
sp1, aval1 = run_sandpile(N=64, n_grains=100000, warmup=10000, seed=123)
aval1_pos = aval1[aval1 > 0]
print(f"Non-zero avalanches: {len(aval1_pos)}")

# MLE fit
s_min, tau_mle, ks = find_s_min_ks(aval1_pos)
tau_s1, sig_s1 = mle_power_law(aval1_pos, s_min)
print(f"MLE: τ = {tau_s1:.3f} ± {sig_s1:.3f} (s_min = {s_min}, KS = {ks:.4f})")

# Naive least-squares on log-log (for comparison)
c, d = log_binned_histogram(aval1_pos, n_bins=30)
# Fit only above s_min
mask_fit = c >= s_min
if mask_fit.sum() > 3:
    slope_ls, intercept_ls, r_ls, _, _ = linregress(np.log10(c[mask_fit]),
                                                     np.log10(d[mask_fit]))
    tau_ls = -slope_ls
    print(f"Least-squares: τ = {tau_ls:.3f} (R² = {r_ls**2:.4f})")
print(f"Literature: τ ≈ 1.27")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

ax1.loglog(c, d, 'o', color=COLOR_A, markersize=4, alpha=0.7)
s_line = np.logspace(np.log10(s_min), np.log10(aval1_pos.max()), 50)
idx_n = np.argmin(np.abs(c - s_min))
C_norm = d[idx_n] * c[idx_n]**tau_s1
ax1.plot(s_line, C_norm * s_line**(-tau_s1), '--', color=COLOR_B, linewidth=2,
         label=f'MLE: τ = {tau_s1:.2f}')
if mask_fit.sum() > 3:
    ax1.plot(s_line, 10**intercept_ls * s_line**slope_ls, ':', color=COLOR_GREEN,
             linewidth=2, label=f'LSQ: τ = {tau_ls:.2f}')
ax1.axvline(x=s_min, color='white', linestyle=':', alpha=0.3)
ax1.set_xlabel('Avalanche size s')
ax1.set_ylabel('P(s)')
ax1.set_title('P(s) with both fits', fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3, which='both')

# CCDF (complementary cumulative distribution)
sorted_aval = np.sort(aval1_pos)[::-1]
ccdf = np.arange(1, len(sorted_aval) + 1) / len(sorted_aval)
ax2.loglog(sorted_aval, ccdf, '.', color=COLOR_A, markersize=1, alpha=0.5)
# For power law with exponent tau, CCDF ~ s^(1-tau)
s_ref = np.logspace(0, np.log10(sorted_aval.max()), 50)
ax2.plot(s_ref, (s_ref / s_min)**(1 - tau_s1), '--', color=COLOR_B,
         linewidth=2, label=f'τ = {tau_s1:.2f}')
ax2.set_xlabel('Avalanche size s')
ax2.set_ylabel('P(S ≥ s)')
ax2.set_title('CCDF (less noisy than PDF)', fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print("\nNote: MLE is more reliable. Least-squares on log-log can be biased")
print("by the choice of bins and gives too much weight to rare large events.")

### Solution 2: Finite-Size Scaling

In [ ]:
grid_sizes = [16, 32, 64, 128]
colors_fss = [COLOR_A, COLOR_B, COLOR_GREEN, COLOR_PURPLE]
all_avals = {}
s_maxes = []

for N, col in zip(grid_sizes, colors_fss):
    n_grains = max(50000, N * N * 30)
    warmup = N * N * 5
    print(f"N = {N}: running {n_grains} grains (warmup {warmup})...")
    _, avals = run_sandpile(N, n_grains=n_grains, warmup=warmup, seed=42)
    all_avals[N] = avals[avals > 0]
    s_maxes.append(all_avals[N].max())
    tau_n, sig_n = mle_power_law(all_avals[N], s_min=2)
    print(f"  n_avals = {len(all_avals[N])}, max = {s_maxes[-1]}, τ = {tau_n:.3f}")

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(17, 5))

# Raw distributions
for N, col in zip(grid_sizes, colors_fss):
    c, d = log_binned_histogram(all_avals[N], n_bins=30)
    ax1.loglog(c, d, 'o-', color=col, markersize=3, linewidth=1,
               label=f'N = {N}')

ax1.set_xlabel('Avalanche size s')
ax1.set_ylabel('P(s)')
ax1.set_title('P(s) for different N', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3, which='both')

# s_max vs N (estimate D)
ax2.loglog(grid_sizes, s_maxes, 'o-', color=COLOR_B, markersize=8, linewidth=2)
slope_D, intercept_D, r_D, _, _ = linregress(np.log(grid_sizes), np.log(s_maxes))
N_fit = np.linspace(12, 150, 50)
ax2.plot(N_fit, np.exp(intercept_D) * N_fit**slope_D, '--', color=COLOR_A,
         linewidth=2, label=f'D ≈ {slope_D:.2f}')
ax2.set_xlabel('Grid size N')
ax2.set_ylabel('Max avalanche size')
ax2.set_title(f's_max ∝ N^D', fontweight='bold')
ax2.legend(fontsize=12)
ax2.grid(True, alpha=0.3, which='both')

# Data collapse: P(s) * s^tau vs s / N^D
tau_collapse = 1.27
D_collapse = slope_D
for N, col in zip(grid_sizes, colors_fss):
    c, d = log_binned_histogram(all_avals[N], n_bins=30)
    x_scaled = c / N**D_collapse
    y_scaled = d * c**tau_collapse
    ax3.loglog(x_scaled, y_scaled, 'o', color=col, markersize=3, alpha=0.7,
               label=f'N = {N}')

ax3.set_xlabel(r's / N$^D$')
ax3.set_ylabel(r'P(s) · s$^τ$')
ax3.set_title(f'Data collapse (τ={tau_collapse}, D≈{D_collapse:.1f})',
              fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print(f"\nEstimated fractal dimension D ≈ {slope_D:.2f}")
print(f"Literature: D ≈ 2.7 for 2D BTW sandpile")

### Solution 3: Transient to Criticality

In [ ]:
# Measure warmup time for different N
warmup_times = []
grid_sizes_3 = [16, 24, 32, 48, 64]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

for N in grid_sizes_3:
    sp = Sandpile(N)
    heights = []
    n_total = N * N * 25

    for i in range(n_total):
        sp.step()
        if i % (N * N // 4) == 0:
            heights.append(sp.mean_height())

    # Estimate warmup: when height reaches 90% of final
    h_final = np.mean(heights[-10:])
    threshold = 0.9 * h_final
    warmup_idx = np.argmax(np.array(heights) >= threshold)
    t_w = warmup_idx * (N * N // 4)
    warmup_times.append(t_w)

    ax1.plot(np.arange(len(heights)) * (N * N // 4) / (N * N),
             heights, linewidth=1.5, label=f'N = {N}')

ax1.set_xlabel('Grains / N²')
ax1.set_ylabel('Mean height ⟨z⟩')
ax1.set_title('Rescaled transient', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Warmup scaling
ax2.loglog(grid_sizes_3, warmup_times, 'o-', color=COLOR_B, markersize=8)
slope_w, intercept_w, _, _, _ = linregress(
    np.log(grid_sizes_3), np.log(np.array(warmup_times) + 1)
)
N_line = np.linspace(10, 80, 50)
ax2.plot(N_line, np.exp(intercept_w) * N_line**slope_w, '--', color=COLOR_A,
         linewidth=2, label=f'slope ≈ {slope_w:.2f}')
ax2.set_xlabel('Grid size N')
ax2.set_ylabel('Warmup time t_w')
ax2.set_title('Warmup scaling', fontweight='bold')
ax2.legend(fontsize=12)
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print(f"Warmup scaling exponent: t_w ∝ N^{slope_w:.1f}")
print(f"Expected: t_w ∝ N² (time to fill the grid)")

### Solution 4: Avalanche Duration and Scaling

In [ ]:
class SandpileExtended(Sandpile):
    """Sandpile that also records duration and area."""

    def step_extended(self):
        """Return (size, duration, area)."""
        self.add_grain()
        total_topplings = 0
        duration = 0
        toppled_sites = np.zeros((self.N, self.N), dtype=bool)

        while True:
            unstable = self.grid >= self.z_crit
            if not unstable.any():
                break
            duration += 1
            total_topplings += unstable.sum()
            toppled_sites |= unstable

            self.grid[unstable] -= self.z_crit
            self.grid[1:, :]  += unstable[:-1, :]
            self.grid[:-1, :] += unstable[1:, :]
            self.grid[:, 1:]  += unstable[:, :-1]
            self.grid[:, :-1] += unstable[:, 1:]

        area = toppled_sites.sum()
        return total_topplings, duration, area


# Collect extended data
print("Running extended sandpile N=64...")
np.random.seed(42)
sp_ext = SandpileExtended(64)

# Warmup
for _ in range(20000):
    sp_ext.step()

sizes, durations, areas = [], [], []
for _ in range(80000):
    s, t, a = sp_ext.step_extended()
    if s > 0:
        sizes.append(s)
        durations.append(t)
        areas.append(a)

sizes = np.array(sizes)
durations = np.array(durations)
areas = np.array(areas)
print(f"Collected {len(sizes)} non-zero avalanches")

# Plot distributions
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for data, label, color, ax in zip(
    [sizes, durations, areas],
    ['Size s', 'Duration t', 'Area a'],
    [COLOR_A, COLOR_B, COLOR_GREEN],
    axes
):
    c, d = log_binned_histogram(data, n_bins=25)
    ax.loglog(c, d, 'o', color=color, markersize=4, alpha=0.7)
    # MLE fit
    tau_fit, sig_fit = mle_power_law(data, s_min=2)
    s_line = np.logspace(np.log10(2), np.log10(data.max()), 50)
    idx_n = np.argmin(np.abs(c - 2))
    if idx_n < len(d):
        C_n = d[idx_n] * c[idx_n]**tau_fit
        ax.plot(s_line, C_n * s_line**(-tau_fit), '--', color='white',
                linewidth=2, label=f'τ = {tau_fit:.2f}')
    ax.set_xlabel(label)
    ax.set_ylabel(f'P({label[0].lower()})')
    ax.set_title(f'{label} distribution', fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

# Size vs duration scaling
fig, ax = plt.subplots(figsize=(8, 6))

# Bin by duration and compute mean size
unique_t = np.unique(durations)
mean_s_by_t = []
t_vals = []
for t in unique_t:
    mask = durations == t
    if mask.sum() >= 5:
        mean_s_by_t.append(np.mean(sizes[mask]))
        t_vals.append(t)

t_vals = np.array(t_vals)
mean_s_by_t = np.array(mean_s_by_t)

ax.loglog(t_vals, mean_s_by_t, 'o', color=COLOR_A, markersize=5)

# Fit s ~ t^gamma
mask_fit = t_vals >= 3
if mask_fit.sum() > 3:
    gamma_slope, gamma_int, r_g, _, _ = linregress(
        np.log(t_vals[mask_fit]), np.log(mean_s_by_t[mask_fit])
    )
    t_line = np.logspace(0, np.log10(t_vals.max()), 50)
    ax.plot(t_line, np.exp(gamma_int) * t_line**gamma_slope, '--',
            color=COLOR_B, linewidth=2,
            label=f'γ ≈ {gamma_slope:.2f}')

ax.set_xlabel('Duration t', fontsize=13)
ax.set_ylabel('⟨s⟩(t)', fontsize=13)
ax.set_title('Size–Duration Scaling: s ∝ t^γ', fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

# Check scaling relation
tau_s, _ = mle_power_law(sizes, s_min=2)
tau_t, _ = mle_power_law(durations, s_min=2)
gamma_predicted = (tau_t - 1) / (tau_s - 1)

print(f"\nExponents:")
print(f"  τ_s = {tau_s:.3f}")
print(f"  τ_t = {tau_t:.3f}")
print(f"  γ (measured) = {gamma_slope:.3f}")
print(f"  γ (predicted from scaling) = (τ_t - 1)/(τ_s - 1) = {gamma_predicted:.3f}")

---
## Summary

**What we learned today:**

1. **SOC** = criticality as an attractor of the dynamics, not a tuned parameter value. The three ingredients: slow driving + threshold dynamics + boundary dissipation.

2. **The BTW sandpile** is the simplest SOC model: add grains, topple above threshold, lose grains at boundaries → power-law avalanches.

3. **Power laws** $P(s) \propto s^{-\tau}$ are the signature of scale-free critical behavior. The exponent $\tau \approx 1.27$ for avalanche sizes in the 2D BTW model.

4. **Fitting power laws** requires MLE, not naive log-log regression (Clauset et al. 2009).

5. The sandpile is a **threshold CA**, implemented with the same NumPy array-slicing tricks as Weeks 1–2 (no Mesa needed — no agents, no decisions).

6. **Finite-size scaling** and **data collapse** connect the sandpile to the universal framework of critical phenomena.

**Weeks 1–4 complete:** We've built a foundation from emergence (CA) → agency (Schelling/Mesa) → self-organized criticality (sandpile). **Next: Reinforcement Learning** — agents that learn from experience.